# 🎬 YOLOv8n — Video Inference
> Run your trained model on any video file, webcam, or RTSP stream.

**Modes supported:**
- 📁 Local video file (mp4, avi, mov …)
- 📷 Webcam (index 0, 1 …)
- 🌐 RTSP / HTTP stream URL
- 🖼️ Single image or image folder

---

## 📦 Step 1 — Install / Import

In [ ]:
!pip install ultralytics --quiet

import cv2, time, torch
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {device}")

## ⚙️ Step 2 — Configuration

In [ ]:
# ── CONFIGURE THESE ────────────────────────────────────────────────────────────

# Path to your trained weights (from training notebook)
WEIGHTS = "runs/detect/yolov8n_custom/weights/best.pt"
# Or use a pretrained model directly:
# WEIGHTS = "yolov8n.pt"

# Input source — pick ONE:
SOURCE  = "your_video.mp4"   # local video file
# SOURCE = 0                 # webcam (0 = default camera)
# SOURCE = "rtsp://..."      # RTSP stream
# SOURCE = "https://..."     # HTTP video URL

# Detection settings
CONF_THRESH = 0.25    # confidence threshold (0–1)
IOU_THRESH  = 0.45    # NMS IoU threshold
IMG_SIZE    = 640     # inference resolution

# Output
SAVE_VIDEO  = True                  # save annotated output video
OUTPUT_PATH = "output_annotated.mp4" # output file name
SHOW_FPS    = True                   # overlay FPS counter on frames
MAX_FRAMES  = None                   # None = process entire video; set e.g. 300 to limit
# ────────────────────────────────────────────────────────────────────────────────

print(f"Weights : {WEIGHTS}")
print(f"Source  : {SOURCE}")

## 🔍 Step 3 — Load Model

In [ ]:
model = YOLO(WEIGHTS)
model.to(device)

class_names = model.names
print(f"✅ Model loaded — {len(class_names)} classes:")
for idx, name in class_names.items():
    print(f"   [{idx}] {name}")

## ▶️ Step 4 — Run Inference on Video

In [ ]:
def draw_fps(frame, fps):
    """Overlay FPS in top-left corner."""
    label = f"FPS: {fps:.1f}"
    cv2.rectangle(frame, (8, 8), (130, 38), (0, 0, 0), -1)
    cv2.putText(frame, label, (12, 30), cv2.FONT_HERSHEY_SIMPLEX,
                0.75, (0, 255, 0), 2, cv2.LINE_AA)
    return frame


def run_video_inference(
    model, source, conf, iou, imgsz,
    save_video=True, output_path="output.mp4",
    show_fps=True, max_frames=None
):
    cap = cv2.VideoCapture(source if isinstance(source, str) else source)

    if not cap.isOpened():
        raise RuntimeError(f"❌ Cannot open source: {source}")

    fps_src = cap.get(cv2.CAP_PROP_FPS) or 30
    w       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video  : {w}×{h} @ {fps_src:.1f} fps  |  {total} frames total")

    writer = None
    if save_video:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(output_path, fourcc, fps_src, (w, h))
        print(f"Output : {output_path}")

    frame_count = 0
    prev_time   = time.time()
    stats       = {name: 0 for name in model.names.values()}

    print("\nProcessing… (Ctrl+C to stop early)")

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if max_frames and frame_count >= max_frames:
                break

            # ── Inference ──────────────────────────────────────────────────────
            results = model.predict(
                frame,
                conf=conf,
                iou=iou,
                imgsz=imgsz,
                verbose=False,
                device=device,
            )[0]

            # ── Annotate ───────────────────────────────────────────────────────
            annotated = results.plot()

            # Accumulate detection counts
            for box in results.boxes:
                cls_name = model.names[int(box.cls)]
                stats[cls_name] += 1

            # FPS
            now  = time.time()
            fps  = 1.0 / (now - prev_time + 1e-6)
            prev_time = now

            if show_fps:
                annotated = draw_fps(annotated, fps)

            if writer:
                writer.write(annotated)

            frame_count += 1

            # Progress every 50 frames
            if frame_count % 50 == 0:
                pct = f"{frame_count/total*100:.1f}%" if total else f"{frame_count} frames"
                print(f"  [{pct}]  frame {frame_count}  |  FPS {fps:.1f}")

    except KeyboardInterrupt:
        print("\n⚠️  Interrupted by user.")

    finally:
        cap.release()
        if writer:
            writer.release()

    print(f"\n✅ Done — {frame_count} frames processed.")
    return stats, frame_count


# ── Run ────────────────────────────────────────────────────────────────────────
stats, total_frames = run_video_inference(
    model      = model,
    source     = SOURCE,
    conf       = CONF_THRESH,
    iou        = IOU_THRESH,
    imgsz      = IMG_SIZE,
    save_video = SAVE_VIDEO,
    output_path= OUTPUT_PATH,
    show_fps   = SHOW_FPS,
    max_frames = MAX_FRAMES,
)

## 📊 Step 5 — Detection Statistics

In [ ]:
# Filter out classes with zero detections
active = {k: v for k, v in stats.items() if v > 0}

if not active:
    print("No detections found. Try lowering CONF_THRESH.")
else:
    print("Detection Summary")
    print("=" * 35)
    for cls, count in sorted(active.items(), key=lambda x: -x[1]):
        print(f"  {cls:<20} : {count:>6} detections")
    print(f"  {'TOTAL':<20} : {sum(active.values()):>6}")
    print(f"\n  Frames processed : {total_frames}")
    print(f"  Avg det/frame    : {sum(active.values())/max(total_frames,1):.2f}")

    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = plt.cm.tab20.colors
    labels = list(active.keys())
    values = list(active.values())
    bars = ax.barh(labels, values, color=[colors[i % 20] for i in range(len(labels))])
    ax.bar_label(bars, padding=4, fontsize=10)
    ax.set_xlabel("Total Detections")
    ax.set_title("Detections per Class across Video", fontweight="bold")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 🖼️ Step 6 — Preview Frames from Output Video

In [ ]:
def preview_video_frames(video_path, n_frames=6):
    """Extract and display evenly-spaced frames from the output video."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total == 0:
        print("No frames in video.")
        return

    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    frames  = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append((idx, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))

    cap.release()

    cols = 3
    rows = (len(frames) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
    axes = np.array(axes).flatten()

    for ax, (idx, frame) in zip(axes, frames):
        ax.imshow(frame)
        ax.set_title(f"Frame {idx}", fontsize=9)
        ax.axis("off")

    for ax in axes[len(frames):]:
        ax.axis("off")

    plt.suptitle(f"Preview — {Path(video_path).name}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


if SAVE_VIDEO and Path(OUTPUT_PATH).exists():
    preview_video_frames(OUTPUT_PATH, n_frames=6)
else:
    print("Output video not found — set SAVE_VIDEO=True and re-run Step 4.")

## 🌐 Step 7 — (Optional) Ultralytics One-liner API
> The simplest way to run inference — Ultralytics handles everything automatically.

In [ ]:
# Ultralytics one-liner: auto-saves annotated video to runs/detect/predict/
model.predict(
    source  = SOURCE,
    conf    = CONF_THRESH,
    iou     = IOU_THRESH,
    imgsz   = IMG_SIZE,
    save    = True,      # saves annotated video
    save_txt= True,      # saves detections as .txt (YOLO format)
    save_conf=True,      # include confidence scores in txt
    stream  = True,      # memory-efficient for long videos
    verbose = False,
)

print("✅ Saved to runs/detect/predict/")

---
## 📋 Quick Reference

| Parameter | Default | Effect |
|-----------|---------|--------|
| `CONF_THRESH` | 0.25 | Lower → more detections, more false positives |
| `IOU_THRESH` | 0.45 | Lower → fewer overlapping boxes |
| `IMG_SIZE` | 640 | Higher → more accurate, slower |
| `MAX_FRAMES` | None | Limit processing for quick tests |

### 🎥 Source Examples
```python
SOURCE = "video.mp4"               # local file
SOURCE = 0                          # webcam
SOURCE = "rtsp://192.168.1.10/..." # IP camera
SOURCE = "https://..."              # HTTP stream / YouTube URL
SOURCE = "images/"                  # folder of images
```

### 💡 Tips
- For **real-time webcam** in Colab: use `source=0` and open video with `cv2.imshow` locally instead
- For **long videos**, set `stream=True` in the one-liner to avoid RAM overflow
- Use `half=True` in `.predict()` for **FP16 inference** (2× faster on GPU)